# Feature Engineering

## Package Imports

In [1]:
from pickle import dump, load
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
import numpy as np
import pandas as pd
import skops.io as sio

## Dataset Loading

In [2]:
with open('data/preprocessed/data.pkl', 'rb') as file:
    train_data = load(file)

with open('data/raw/test_data.pkl', 'rb') as file:
    test_data = load(file)

display(train_data.head())
print(f"number of row: {len(train_data)}")
train_data.info()
print('---')
display(test_data.head())
print(f"number of row: {len(test_data)}")
test_data.info()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,EstimatedSalary,Exited
8217,6233,15718242,Wollstonecraft,725,Germany,Female,47,1,104887.43,86622.56,1
1664,548,15720187,Han,479,Germany,Female,30,7,143964.36,41879.99,0
3599,8343,15773876,Tung,655,France,Female,34,3,0.00,159638.77,0
5540,9437,15771000,Powell,684,France,Male,38,4,0.00,75609.84,0
0,747,15787619,Hsieh,844,France,Male,18,2,160980.03,145936.28,0


number of row: 7223
<class 'pandas.core.frame.DataFrame'>
Index: 7223 entries, 8217 to 5431
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        7223 non-null   int64  
 1   CustomerId       7223 non-null   int64  
 2   Surname          7223 non-null   object 
 3   CreditScore      7223 non-null   int64  
 4   Geography        7223 non-null   object 
 5   Gender           7223 non-null   object 
 6   Age              7223 non-null   int64  
 7   Tenure           7223 non-null   int64  
 8   Balance          7223 non-null   float64
 9   EstimatedSalary  7223 non-null   float64
 10  Exited           7223 non-null   int64  
dtypes: float64(2), int64(6), object(3)
memory usage: 677.2+ KB
---


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,EstimatedSalary,Exited
2223,6095,15575623,Simpson,589,France,Female,31,10,110635.32,148218.86,0
9735,8712,15673995,Tu,516,Spain,Female,65,9,102541.10,181490.42,0
57,2833,15758171,Tien,582,France,Male,20,4,0.00,55763.66,0
9243,6195,15794273,Hand,604,France,Female,56,0,62732.65,124954.56,0
3428,4601,15577985,Chinomso,574,France,Female,34,5,112324.45,17993.43,0


number of row: 2500
<class 'pandas.core.frame.DataFrame'>
Index: 2500 entries, 2223 to 6620
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        2500 non-null   int64  
 1   CustomerId       2500 non-null   int64  
 2   Surname          2500 non-null   object 
 3   CreditScore      2500 non-null   int64  
 4   Geography        2500 non-null   object 
 5   Gender           2500 non-null   object 
 6   Age              2500 non-null   int64  
 7   Tenure           2500 non-null   int64  
 8   Balance          2500 non-null   float64
 9   EstimatedSalary  2500 non-null   float64
 10  Exited           2500 non-null   int64  
dtypes: float64(2), int64(6), object(3)
memory usage: 234.4+ KB


## Feature Engineering Strategy

* We drop `RowNumber`, `CustomerId`, and `Surname` since row number and id are unique identifier, while Surnames are for more marketing purposes.
* Again, `Exited` here is the main target
* `Geography` and `Gender` are considered as categorical feature. Here we use `OrdinalEncoding` in order to use `HistGradientBoostingClassifier` native categorical data support without breaking `shap`
* No need to deal with numerical features thanks to tree-based ensembel model


## Separate Features and Target

In [3]:
X_train = train_data.copy().drop('Exited', axis=1)
y_train = train_data['Exited'].copy()
X_train = X_train.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

display(X_train.head())
display(y_train.head())

X_test = test_data.copy().drop('Exited', axis=1)
y_test = test_data['Exited'].copy()
X_test = X_test.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

,CreditScore,Geography,Gender,Age,Tenure,Balance,EstimatedSalary
8217,725,Germany,Female,47,1,104887.43,86622.56
1664,479,Germany,Female,30,7,143964.36,41879.99
3599,655,France,Female,34,3,0.00,159638.77
5540,684,France,Male,38,4,0.00,75609.84
0,844,France,Male,18,2,160980.03,145936.28


8217    1
1664    0
3599    0
5540    0
0       0
Name: Exited, dtype: int64

## Train-Test Split

Split already done on `download_data.py`

## Label Encoding

### Obtain unique values on X_train

In [4]:
## get values existing on X_train's Geography and Gender
X_geo = X_train['Geography'].unique()
X_gen = X_train['Gender'].unique()
print(X_geo)
print(X_gen)


['Germany' 'France' 'Spain']
['Female' 'Male']


### Set-up and Fit OrdinalEncoder

In [5]:
geo_class = ['France', 'Germany', 'Spain']
g_class = ['Female', 'Male']

X_ord = X_train[['Geography', 'Gender']]
enc_cat = [geo_class, g_class]

enc = OrdinalEncoder(categories=enc_cat, ## enable pre-defined categories
                     handle_unknown='use_encoded_value', ## if a category was not learned at fitting...
                     unknown_value=np.nan) ## assume NaN

enc.fit(X_ord)
display(enc)

,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute.","[['France', 'Germany', ...], ['Female', 'Male']]"
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'use_encoded_value'}, default='error'When set to 'error' an error will be raised in case an unknowncategorical feature is present during transform. When set to'use_encoded_value', the encoded value of unknown categories will beset to the value given for the parameter `unknown_value`. In:meth:`inverse_transform`, an unknown category will be denoted as None... versionadded:: 0.24",'use_encoded_value'
,"unknown_value unknown_value: int or np.nan, default=NoneWhen the parameter handle_unknown is set to 'use_encoded_value', thisparameter is required and will set the encoded value of unknowncategories. It has to be distinct from the values used to encode any ofthe categories in `fit`. If set to np.nan, the `dtype` parameter mustbe a float dtype... versionadded:: 0.24",nan
,"encoded_missing_value encoded_missing_value: int or np.nan, default=np.nanEncoded value of missing categories. If set to `np.nan`, then the `dtype`parameter must be a float dtype... versionadded:: 1.1",nan
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.3 Read more in the :ref:`User Guide `.",None
,"max_categories max_categories: int, default=NoneSpecifies an upper limit to the number of output categories for each inputfeature when considering infrequent categories. If there are infrequentcategories, `max_categories` includes the category representing theinfrequent categories along with the frequent categories. If `None`,there is no limit to the number of output features.`max_categories` do **not** take into account missing or unknowncategories. Setting `unknown_value` or `encoded_missing_value` to aninteger will increase the number of unique integer codes by one each.This can result in up to `max_categories + 2` integer codes... versionadded:: 1.3 Read more in the :ref:`User Guide `.",None


In [6]:
## transform X_train, X_valid, and X_test to do things
def enc_transform(X, enc):
    cat_feat = enc.get_feature_names_out()
    X_ord = X[cat_feat]
    X_enc = enc.transform(X_ord)
    X_enc = pd.DataFrame(X_enc, index=X.index,
                         columns=enc.get_feature_names_out())
    
    X[cat_feat] = X_enc
    return X

    ## legacy from my old code: roughly drop old column and redefine
    # X = X.drop(cat_feat, axis=1)
    # X = X.merge(X_enc, how='inner',
    #             left_index=True, right_index=True) ## use index as keys

X_train = enc_transform(X_train, enc)
X_test = enc_transform(X_test, enc)

print(f"Training data, row: {len(X_train)}")
display(X_train.head())
print(f"Test data, row: {len(X_test)}")
display(X_test.head())


Training data, row: 7223


,CreditScore,Geography,Gender,Age,Tenure,Balance,EstimatedSalary
8217,725,1.0,0.0,47,1,104887.43,86622.56
1664,479,1.0,0.0,30,7,143964.36,41879.99
3599,655,0.0,0.0,34,3,0.00,159638.77
5540,684,0.0,1.0,38,4,0.00,75609.84
0,844,0.0,1.0,18,2,160980.03,145936.28


Test data, row: 2500


,CreditScore,Geography,Gender,Age,Tenure,Balance,EstimatedSalary
2223,589,0.0,0.0,31,10,110635.32,148218.86
9735,516,2.0,0.0,65,9,102541.10,181490.42
57,582,0.0,1.0,20,4,0.00,55763.66
9243,604,0.0,0.0,56,0,62732.65,124954.56
3428,574,0.0,0.0,34,5,112324.45,17993.43


## Saving Data

In [7]:
if not os.path.exists('data/feature_eng'):
    os.makedirs('data/feature_eng')

with open('data/feature_eng/X_train.pkl', 'wb') as file:
    dump(X_train, file)

with open('data/feature_eng/y_train.pkl', 'wb') as file:
    dump(y_train, file)

with open('data/feature_eng/X_test.pkl', 'wb') as file:
    dump(X_test, file)

with open('data/feature_eng/y_test.pkl', 'wb') as file:
    dump(y_test, file)

obj_skops = sio.dump(enc, "data/feature_eng/encoder.skops")